In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

!pip -q install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
df = pd.read_csv("/kaggle/input/airlinedatasetcleaned/AirlineScrappedReview_Cleaned.csv")

# Prepare text (handle NaNs / whitespace)
text_col = "Review_content" if "Review_content" in df.columns else "review_content"
if text_col not in df.columns:
    raise KeyError(f"Couldn't find a review text column. Expected 'Review_Content' (or 'review_content'). Columns found: {list(df.columns)}")

df[text_col] = df[text_col].astype(str).fillna("").str.strip()

# Create the analyzer and add ONE sentiment column (label only)
analyzer = SentimentIntensityAnalyzer()

def label_sentiment(text: str) -> str:
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["Sentiment"] = df[text_col].apply(label_sentiment)
df["Sentiment_Score"] = df[text_col].apply(lambda t: analyzer.polarity_scores(t)['compound'])


# Save the augmented file 
out_path = "AirlineScrappedReview_Sentiment.csv"
df.to_csv(out_path, index=False)
print(f"Saved with Sentiment column ➜ {out_path}")

In [ ]:
# --- Clean 'Unknown' in Traveller_Type and Class by imputing the mode (most frequent non-Unknown) ---
for col in ['Traveller_Type', 'Class']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        known_mask = df[col].str.casefold() != 'unknown'
        if known_mask.any():
            mode_val = df.loc[known_mask, col].mode().iloc[0]
            df.loc[~known_mask, col] = mode_val

# --- Save cleaned CSV ---
out_path = "AirlineScrappedReview_Cleaned_ModeImputed.csv"
df.to_csv(out_path, index=False)
print(f"Saved cleaned file to: {out_path}")


In [ ]:
#Group by traveler type and seat type
grouped = df.groupby(['Traveller_Type', 'Class'])['Rating'].mean().reset_index()

# Sort to identify highest and lowest
highest = grouped.loc[grouped['Rating'].idxmax()]
lowest = grouped.loc[grouped['Rating'].idxmin()]

print("Highest Rated Combination:")
print(highest)

print("\nLowest Rated Combination:")
print(lowest)

In [ ]:
df.head(20)

In [ ]:
df2 = pd.read_csv("/kaggle/input/airline-passangers-booking-data/Passanger_booking_data.csv")

# Clean Route column (remove NaNs/blank/whitespace-only)
df2['route'] = df2['route'].astype(str).str.strip()
df2 = df2[df2['route'].notna() & (df2['route'] != '')]

# Compute Top 10 routes 
top_routes = df2['route'].value_counts().head(10).reset_index()
top_routes.columns = ['route', 'Bookings']

# Print results 
print("Top 10 Most Popular Flight Routes:")
print(top_routes.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=top_routes, x='Bookings', y='route')
plt.title("Top 10 Most Popular Flight Routes")
plt.xlabel("Number of Bookings")
plt.ylabel("Route")
plt.tight_layout()
plt.show()

In [ ]:
# Convert to numeric (if stored as string)
df2['flight_hour'] = pd.to_numeric(df2['flight_hour'], errors='coerce')

# Drop missing hours if any
df2 = df2.dropna(subset=['flight_hour'])

# Group by flight hour
hourly_bookings = df2['flight_hour'].value_counts().sort_index()

print("Booking distribution by flight hour:")
print(hourly_bookings)

# Visualization
plt.figure(figsize=(10,6))
sns.barplot(x=hourly_bookings.index, y=hourly_bookings.values, palette='viridis')
plt.title("Distribution of Bookings Across Flight Hours")
plt.xlabel("Flight Hour (0–23)")
plt.ylabel("Number of Bookings")
plt.xticks(range(0,24))
plt.show()

In [ ]:
# Check if the column exists
if 'Passanger_Name' not in df.columns:
    raise KeyError(f"Couldn't find a 'Name' column. Columns found: {list(df.columns)}")

# Fill null or missing names with "Amina Tamer"
df['Passanger_Name'] = df['Passanger_Name'].fillna('Amina Tamer')

# Optional: also replace empty strings or spaces with "Amina Tamer"
df['Passanger_Name'] = df['Passanger_Name'].replace(r'^\s*$', 'Amina Tamer', regex=True)

# Save the cleaned CSV
out_path = "AirlineScrappedReview_FilledNames.csv"
df.to_csv(out_path, index=False)

print(f"Saved cleaned file with missing names replaced ➜ {out_path}")

In [ ]:
df.info()

In [ ]:
cols_to_drop = [
    'Flying_Date',
    'Start_Location',
    'End_Location',
    'Layover_Route',
    'Start_Latitude',
    'Start_Longitude',
    'Start_Address',
    'End_Latitude',
    'End_Longitude',
    'End_Address'
]

# Keep only columns that actually exist to avoid KeyError
existing_to_drop = [c for c in cols_to_drop if c in df.columns]
if existing_to_drop:
    df = df.drop(columns=existing_to_drop)
    print(f"Dropped columns: {existing_to_drop}")
else:
    print("No matching columns found to drop.")

# Re-save (same filename as before, or change if you want a separate version)
out_path = "AirlineScrappedReview_droppedcols.csv"  # or your previous out_path
df.to_csv(out_path, index=False)
print(f"Saved cleaned file (after drops) ➜ {out_path}")

In [ ]:
# 1) Define explicit mappings
class_map = {
    "Economy Class": 0,
    "Premium Economy": 1,
    "Business Class": 2,
    "First Class": 3,
}

verified_map = {
    "Not Verified": 0,
    "Trip Verified": 1,
}

sentiment_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2,
}

trav_map = {
    "Solo Leisure": 0,
    "Couple Leisure": 1,
    "Family Leisure": 2,
    "Business": 3,
    "Various": 4,
}

# 2) Apply mappings (keep originals; add *_enc columns)
if "Class" in df.columns:
    df["Class_enc"] = df["Class"].map(class_map)

if "Verified" in df.columns:
    df["Verified_enc"] = df["Verified"].map(verified_map)

if "Sentiment" in df.columns:
    df["Sentiment_enc"] = df["Sentiment"].map(sentiment_map)

if "Traveller_Type" in df.columns:
    df["Traveller_Type_enc"] = df["Traveller_Type"].map(trav_map)

out_path = "AirlineScrappedReview_Encoded.csv"  
df.to_csv(out_path, index=False)
print(f"Saved CSV with encoded columns ➜ {out_path}")

In [ ]:
# === Setup ===
import pandas as pd
import numpy as np

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# NN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42

# === 1) Load data & build target ===
csv_path = "/kaggle/working/AirlineScrappedReview_Encoded.csv"
df = pd.read_csv(csv_path)

# Ensure needed columns exist
required_cols = {"Rating"}
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Missing required columns: {missing}. Available: {list(df.columns)}")

# Build binary target: Satisfied (>=5) vs Dissatisfied (<5)
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df.dropna(subset=["Rating"]).reset_index(drop=True)
df["Satisfied"] = (df["Rating"] >= 5).astype(int)

# === 2) Choose two feature sets (EDIT THESE AS YOU LIKE) ===
ALL_ALLOWED = ["Verified_enc", "Traveller_Type_enc", "Class_enc", "Sentiment_enc"]

FEATURE_SET_A = ["Traveller_Type_enc", "Class_enc", "Sentiment_enc"]                      
FEATURE_SET_B = ["Verified_enc", "Traveller_Type_enc", "Class_enc", "Sentiment_enc"]  

for needed in set(FEATURE_SET_A + FEATURE_SET_B):
    if needed not in df.columns:
        raise KeyError(f"Feature '{needed}' not found. Available: {list(df.columns)}")

# === 3) Utilities ===
def split_scale(X, y, test_size=0.2, random_state=RANDOM_STATE):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler

def eval_metrics(y_true, y_pred):
    return {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
    }

def train_logreg(X_train_scaled, y_train):
    # Simple Logistic Regression (L2, balanced class weights)
    clf = LogisticRegression(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        max_iter=1000
    )
    clf.fit(X_train_scaled, y_train)
    return clf

def make_ffnn(input_dim, hidden_units=16, dropout=0.1):
    # model = keras.Sequential([
    #     layers.Input(shape=(input_dim,)),
    #     layers.Dense(64, activation='relu'),
    #     layers.Dense(32, activation='relu'),
    #     layers.Dense(32, activation='relu'),
    #     layers.Dense(16, activation='relu'),
    #     layers.Dense(16, activation='relu'),
    #     layers.Dense(8, activation='relu'),
    #     layers.Dense(1, activation="sigmoid")
    # ])
    # model = keras.Sequential([
    #     layers.Input(shape=(input_dim,)),
    #     layers.Dense(64, activation='relu'),
    #     layers.Dense(32, activation='relu'),
    #     layers.Dense(16, activation='relu'),
    #     layers.Dense(8, activation='relu'),
    #     layers.Dense(1, activation="sigmoid")
    # ])
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='selu', kernel_initializer='lecun_normal'),
        layers.AlphaDropout(0.1),
        layers.Dense(32, activation='selu', kernel_initializer='lecun_normal'),
        layers.AlphaDropout(0.1),
        layers.Dense(16, activation='selu', kernel_initializer='lecun_normal'),
        layers.AlphaDropout(0.05),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def train_ffnn(X_train_scaled, y_train, X_val_scaled, y_val):
    # Handle potential imbalance
    # Compute class weights: {0: w0, 1: w1}
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)
    total = len(y_train)
    # Inverse-frequency style
    class_weight = {
        0: total / (2.0 * neg),
        1: total / (2.0 * pos),
    }

    model = make_ffnn(X_train_scaled.shape[1], hidden_units=16, dropout=0.1)
    cb = [
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss")
    ]
    model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=80,
        batch_size=64,
        class_weight=class_weight,
        verbose=1,
        callbacks=cb
    )
    return model

def run_experiment(feature_names, label=""):
    X = df[feature_names].copy()
    y = df["Satisfied"].copy()

    # Replace any missing feature values with most frequent (simple imputation)
    for c in feature_names:
        if X[c].isna().any():
            mode_val = X[c].mode(dropna=True)
            fill_val = mode_val.iloc[0] if not mode_val.empty else 0
            X[c] = X[c].fillna(fill_val)

    X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler = split_scale(X, y)

    # --- Logistic Regression ---
    logreg = train_logreg(X_train_scaled, y_train)
    y_pred_lr = (logreg.predict_proba(X_test_scaled)[:, 1] >= 0.5).astype(int)
    metrics_lr = eval_metrics(y_test, y_pred_lr)

    # --- FFNN ---
    ffnn = train_ffnn(X_train_scaled, y_train, X_test_scaled, y_test)
    y_pred_nn = (ffnn.predict(X_test_scaled, verbose=0).ravel() >= 0.5).astype(int)
    metrics_nn = eval_metrics(y_test, y_pred_nn)

    # Collect
    result_rows = []
    result_rows.append({
        "feature_set": label,
        "model": "LogisticRegression",
        **metrics_lr
    })
    result_rows.append({
        "feature_set": label,
        "model": "ShallowFFNN",
        **metrics_nn
    })
    return pd.DataFrame(result_rows)

# === 4) Run both feature sets ===
results_a = run_experiment(FEATURE_SET_A, label=f"A: {FEATURE_SET_A}")
results_b = run_experiment(FEATURE_SET_B, label=f"B: {FEATURE_SET_B}")

all_results = pd.concat([results_a, results_b], ignore_index=True)
display(all_results)

# Save metrics
all_results.to_csv("model_metrics.csv", index=False)
print("Saved metrics to model_metrics.csv")